In [1]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('..'))
import matplotlib.pyplot as plt
from scripts import nodes as n
from scripts import elements as e
from scripts import material_params as mat
from scipy.linalg import eigh
import plotly.graph_objects as go
from scripts import FDD as fdd

In [2]:
nodes = []
nodal_values = np.loadtxt('../text_files/nodes_minimal_model.txt', delimiter=',')
for i in range(nodal_values.shape[0]):
  nodes.append(n.nodes(nodal_values[i, 0], nodal_values[i, 1], nodal_values[i, 2]))

## **Constructing parameter space for natural frequencies and mode shapes**

**Chosen parameters for this parameters space are as follows:**

- Axial stiffness of the fenders $k_f$
- Moment of inertia of the primary truss $I_{y,truss}$
- In-plane axial stiffness at the locomobile $k_l$

In [ ]:
N = 50

kf_min = 5e6
kf_max = 5e7

Iy_min = 100
Iy_max = 2000

kl_min = 0.2e10
kl_max = 1e10

Iy_wall_min = 100
Iy_wall_max = 2000

h_eq = 22
m_wall = 6455148
L_wall = 237.5
Iy_wall_vals = np.arange(Iy_wall_min, Iy_wall_max, (Iy_wall_max - Iy_wall_min) / N)
Iz_wall_vals = 0.25 * Iy_wall_vals
Ip_wall_vals = Iy_wall_vals + Iz_wall_vals  
It_wall_vals = 0.05 * Iy_wall_vals
A_wall_vals = Iy_wall_vals * 12 / h_eq**2
rho_wall_vals = m_wall / (L_wall * A_wall_vals)

kf_vals = np.arange(kf_min, kf_max, (kf_max-kf_min)/N)
Iy_vals = np.arange(Iy_min, Iy_max, (Iy_max-Iy_min)/N)
kl_vals = np.arange(kl_min, kl_max, (kl_max-kl_min)/N)
Iz_vals = 0.4 * Iy_vals
Ip_vals = Iy_vals + Iz_vals
b  = ((Iz_vals**3) / (Iy_vals* (1/12)**2))**(1/8)
h = Iz_vals / ((1/12)* b**3)
It = (b * h**3 / 3) * (1 - 0.63 * (h / b) * (1 - (h**4 / (12 * b**4)))) *0.06
m = 5596024
L = 237.5
d1 = 0.8 #m
d2 = 1.8 #m
t1 = 0.03 #m
t2 = 0.08 #m
A_vals = 0.008129475143596973 * Iy_vals
rho_truss_update = m / (L * A_vals)
k = 0.08

## **All fixed parameters**

In [4]:
E = 210e9  # Young's modulus in Pascals
nu = 0.3   # Poisson's ratio
G = E / (2 * (1 + nu))  # Shear modulus in Pascals

L_truss_element_y = 15 #m
h_truss = 18 #m



k = 5/6
rho = 7850 #kg/m^3
E=210e9 #Pa
G = E/(2*(1+nu)) #Pa
nu = 0.3 #Poisson's ratio
k_fender =  mat.stiffness_fenders()
Iy_connect, Iz_connect, Ip_connect, It_connect, A_connect = mat.stiffness_connecting_beams()
ep_K_connect = [E, G, A_connect, Iy_connect, Iz_connect, It_connect, k]
ep_m_connect = [rho, A_connect, Iy_connect, Iz_connect, Ip_connect]

h_eq = 22
Iy_wall = 1250
Iz_wall = 0.25 * Iy_wall
Ip_wall = Iy_wall + Iz_wall
It_wall = 0.05 * Iy_wall
A_wall = Iy_wall * 12 / h_eq**2
m_wall = 6455148
L_wall = 237.5
rho_wall = m_wall / (L_wall * A_wall)
ep_K_wall = [E, G, A_wall, Iy_wall, Iz_wall, It_wall, k]
ep_m_wall = [rho_wall, A_wall, Iy_wall, Iz_wall, Ip_wall]



A_eq, Iy_eq, Iz_eq, b_eq, h_eq = mat.stiffness_connecting_truss(d1, d2, t1, t2, h_truss, L_truss_element_y)
Ip_connecting_truss = Iy_eq + Iz_eq 
It_eq = (b_eq * h_eq**3 / 3) * (1 - 0.63 * (h_eq / b_eq) * (1 - (h_eq**4 / (12 * b_eq**4)))) *0.06
L = 237.5

rho_truss = m / (L * A_eq)
ep_K_connecting_truss = [E, G, A_eq, Iy_eq, Iz_eq, It_eq, k]
ep_m_connecting_truss = [rho_truss, A_eq, Iy_eq, Iz_eq, Ip_connect]


# braces
k = 5/6
Iy, Iz, Ip, It, A = mat.stiffness_braces()
ep_K_braces = [E, G, A, Iy, Iz, It, k]
ep_m_braces = [rho, A, Iy, Iz, Ip]

## **Creating the three-parameter set I**

In [ ]:
NN = len(nodes)
dofs = n.degrees_of_freedom(nodes)
DOFS_per_node = 6
fender_dofs = [70, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 128]
k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
kl_dofs = [dofs['dof_75'][0], dofs['dof_75'][1]] 
kr_dofs = [dofs['dof_1'][3], dofs['dof_1'][4], dofs['dof_1'][5]] 
param_space = np.zeros((N, N, N, 6))  # 6 natural frequencies are stored
param_space_eigvecs = np.zeros((N, N, N, NN * DOFS_per_node, 6)) 

for i in range(len(Iy_vals)):
    # the changing truss mass and stiffness matrices
    ep_K = [E, G, A_vals[i], Iy_vals[i], Iz_vals[i], It, k]
    ep_m = [rho_truss_update[i], A_vals[i], Iy_vals[i], Iz_vals[i], Ip_connect]

    elements = []
    element_nodes = np.loadtxt('../text_files/element_nodes.txt', dtype=int)
    for elem in range(element_nodes.shape[0]):
        if element_nodes[elem, 2] == 0:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K, ep_m))
        elif element_nodes[elem, 2] == 1:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_wall, ep_m_wall))
        elif element_nodes[elem, 2] == 2:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_connect, ep_m_connect))
        elif element_nodes[elem, 2] == 3:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_connecting_truss, ep_m_connecting_truss))   
        elif element_nodes[elem, 2] == 4:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_braces, ep_m_braces))
    element_nodes = element_nodes[:, :2] # Remove the column with element type information

    dofs = n.degrees_of_freedom(nodes)

    element_locs = []

    for (nA, nB) in element_nodes:
        dofs_A = dofs[f'dof_{nA}']
        dofs_B = dofs[f'dof_{nB}']
        element_locs.append(np.hstack((dofs_A, dofs_B)))

    for j in range(len(kf_vals)):
        K_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))
        M_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))

        K_locs = []
        M_locs = []

        for elem_loc in range(len(element_locs)):
            K_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-1]
            M_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-2]

        K_global = 0.5 * (K_global + K_global.T)
        M_global = 0.5 * (M_global + M_global.T)

        k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
        for dof in k_fender_dofs:
            K_global[dof, dof] += kf_vals[j]
        
        for k in range(len(kl_vals)):
            for dof in kl_dofs:
                K_global[dof, dof] += kl_vals[k]

            indices_to_remove = np.hstack((dofs['dof_1'][0:3]))
            keep_indices = np.setdiff1d(np.arange(NN * DOFS_per_node), indices_to_remove)
            K_global_reduced = K_global[np.ix_(keep_indices, keep_indices)]
            M_global_reduced = M_global[np.ix_(keep_indices, keep_indices)]
            eigvals_global, eigvecs_global = eigh(K_global_reduced, M_global_reduced)   

            tol = 1e-6
            positive = eigvals_global > tol
            eigvals_global = eigvals_global[positive]
            eigvecs_global = eigvecs_global[:, positive]
            eigvecs_full = e.expand_eigenvectors(eigvecs_global, keep_indices, NN * DOFS_per_node)

            frequencies_rad = np.sqrt(eigvals_global)
            frequencies_hz = frequencies_rad / (2 * np.pi)

            K_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))
            M_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))

            K_locs = []
            M_locs = []

            for elem_loc in range(len(element_locs)):
                K_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-1]
                M_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-2]

            K_global = 0.5 * (K_global + K_global.T)
            M_global = 0.5 * (M_global + M_global.T)

            k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
            for dof in k_fender_dofs:
                K_global[dof, dof] += kf_vals[j]

            param_space[i, j, k, :] = frequencies_hz[:6]
            param_space_eigvecs[i, j, k, :, :] = eigvecs_full[:, :6]

    print(f'Completed iteration {i+1} out of {len(Iy_vals)}')


np.save('param_spaces/param_space.npy', param_space)
np.save('param_spaces/param_space_eigvecs.npy', param_space_eigvecs)

# **Creating the two-parameter set**

In [5]:
NN = len(nodes)
dofs = n.degrees_of_freedom(nodes)
DOFS_per_node = 6
fender_dofs = [70, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 128]
k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
kr_dofs = [dofs['dof_1'][3], dofs['dof_1'][4], dofs['dof_1'][5]] 
param_space = np.zeros((N, N, 6))  # 6 natural frequencies are stored
param_space_eigvecs = np.zeros((N, N, NN * DOFS_per_node, 6)) 

for i in range(len(Iy_vals)):
    # the changing truss mass and stiffness matrices
    ep_K = [E, G, A_vals[i], Iy_vals[i], Iz_vals[i], It, k]
    ep_m = [rho_truss_update[i], A_vals[i], Iy_vals[i], Iz_vals[i], Ip_connect]

    elements = []
    element_nodes = np.loadtxt('../text_files/element_nodes.txt', dtype=int)
    for elem in range(element_nodes.shape[0]):
        if element_nodes[elem, 2] == 0:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K, ep_m))
        elif element_nodes[elem, 2] == 1:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_wall, ep_m_wall))
        elif element_nodes[elem, 2] == 2:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_connect, ep_m_connect))
        elif element_nodes[elem, 2] == 3:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_connecting_truss, ep_m_connecting_truss))   
        elif element_nodes[elem, 2] == 4:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_braces, ep_m_braces))
    element_nodes = element_nodes[:, :2] # Remove the column with element type information

    dofs = n.degrees_of_freedom(nodes)

    element_locs = []

    for (nA, nB) in element_nodes:
        dofs_A = dofs[f'dof_{nA}']
        dofs_B = dofs[f'dof_{nB}']
        element_locs.append(np.hstack((dofs_A, dofs_B)))

    for j in range(len(kf_vals)):
        K_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))
        M_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))

        K_locs = []
        M_locs = []

        for elem_loc in range(len(element_locs)):
            K_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-1]
            M_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-2]

        K_global = 0.5 * (K_global + K_global.T)
        M_global = 0.5 * (M_global + M_global.T)

        k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
        for dof in k_fender_dofs:
            K_global[dof, dof] += kf_vals[j]
        

        indices_to_remove = np.hstack((dofs['dof_1'][0:3], dofs['dof_75'][0:2]))
        keep_indices = np.setdiff1d(np.arange(NN * DOFS_per_node), indices_to_remove)
        K_global_reduced = K_global[np.ix_(keep_indices, keep_indices)]
        M_global_reduced = M_global[np.ix_(keep_indices, keep_indices)]
        eigvals_global, eigvecs_global = eigh(K_global_reduced, M_global_reduced)   

        tol = 1e-6
        positive = eigvals_global > tol
        eigvals_global = eigvals_global[positive]
        eigvecs_global = eigvecs_global[:, positive]
        eigvecs_full = e.expand_eigenvectors(eigvecs_global, keep_indices, NN * DOFS_per_node)

        frequencies_rad = np.sqrt(eigvals_global)
        frequencies_hz = frequencies_rad / (2 * np.pi)


        param_space[i, j, :] = frequencies_hz[:6]
        param_space_eigvecs[i, j, :, :] = eigvecs_full[:, :6]

    print(f'Completed iteration {i+1} out of {len(Iy_vals)}')


np.save('param_spaces/param_space2D.npy', param_space)
np.save('param_spaces/param_space_eigvecs2D.npy', param_space_eigvecs)

Completed iteration 1 out of 100
Completed iteration 2 out of 100
Completed iteration 3 out of 100
Completed iteration 4 out of 100
Completed iteration 5 out of 100
Completed iteration 6 out of 100
Completed iteration 7 out of 100
Completed iteration 8 out of 100
Completed iteration 9 out of 100
Completed iteration 10 out of 100
Completed iteration 11 out of 100
Completed iteration 12 out of 100
Completed iteration 13 out of 100
Completed iteration 14 out of 100
Completed iteration 15 out of 100
Completed iteration 16 out of 100
Completed iteration 17 out of 100
Completed iteration 18 out of 100
Completed iteration 19 out of 100
Completed iteration 20 out of 100
Completed iteration 21 out of 100
Completed iteration 22 out of 100
Completed iteration 23 out of 100
Completed iteration 24 out of 100
Completed iteration 25 out of 100
Completed iteration 26 out of 100
Completed iteration 27 out of 100
Completed iteration 28 out of 100
Completed iteration 29 out of 100
Completed iteration 30 

## **Constructing the three-parameter set II**

In [ ]:
NN = len(nodes)
dofs = n.degrees_of_freedom(nodes)
DOFS_per_node = 6
fender_dofs = [70, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 128]
k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
param_space = np.zeros((N, N, N, 6))  # 6 natural frequencies are stored
param_space_eigvecs = np.zeros((N, N, N, NN * DOFS_per_node, 6)) 
for i in range(len(Iy_vals)):
    # the changing truss mass and stiffness matrices
    ep_K = [E, G, A_vals[i], Iy_vals[i], Iz_vals[i], It, k]
    ep_m = [rho_truss_update[i], A_vals[i], Iy_vals[i], Iz_vals[i], Ip_connect]


    for j in range(len(Iy_wall_vals)):

        ep_K_wall = [E, G, A_wall_vals[j], Iy_wall_vals[j], Iz_wall_vals[j], It_wall_vals[j], k]
        ep_m_wall = [rho_wall_vals[j], A_wall_vals[j], Iy_wall_vals[j], Iz_wall_vals[j], Ip_wall_vals[j]]
        element_nodes = np.loadtxt('../text_files/element_nodes.txt', dtype=int)
        elements = []
        for elem in range(element_nodes.shape[0]):
            if element_nodes[elem, 2] == 0:
                elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K, ep_m))
            elif element_nodes[elem, 2] == 1:
                elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_wall, ep_m_wall))
            elif element_nodes[elem, 2] == 2:
                elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_connect, ep_m_connect))
            elif element_nodes[elem, 2] == 3:
                elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_connecting_truss, ep_m_connecting_truss))   
            elif element_nodes[elem, 2] == 4:
                elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_braces, ep_m_braces))
        element_nodes = element_nodes[:, :2] # Remove the column with element type information

        dofs = n.degrees_of_freedom(nodes)

        element_locs = []

        for (nA, nB) in element_nodes:
            dofs_A = dofs[f'dof_{nA}']
            dofs_B = dofs[f'dof_{nB}']
            element_locs.append(np.hstack((dofs_A, dofs_B)))

        for k in range(len(kf_vals)):
            K_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))
            M_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))

            K_locs = []
            M_locs = []

            for elem_loc in range(len(element_locs)):
                K_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-1]
                M_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-2]

            K_global = 0.5 * (K_global + K_global.T)
            M_global = 0.5 * (M_global + M_global.T)

            k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
            for dof in k_fender_dofs:
                K_global[dof, dof] += kf_vals[k]

            indices_to_remove = np.hstack((dofs['dof_1'][0:3], dofs['dof_75'][0:2]))
            keep_indices = np.setdiff1d(np.arange(NN* DOFS_per_node), indices_to_remove)
            K_global_reduced = K_global[np.ix_(keep_indices, keep_indices)]
            M_global_reduced = M_global[np.ix_(keep_indices, keep_indices)]
            eigvals_global, eigvecs_global = eigh(K_global_reduced, M_global_reduced)


            tol = 1e-6
            positive = eigvals_global > tol
            eigvals_global = eigvals_global[positive]
            eigvecs_global = eigvecs_global[:, positive]

            frequencies_rad = np.sqrt(eigvals_global)
            frequencies_hz = frequencies_rad / (2 * np.pi)
            eigvecs_full = e.expand_eigenvectors(eigvecs_global, keep_indices, NN*DOFS_per_node)
            param_space[i, j, k, :] = frequencies_hz[:6]
            param_space_eigvecs[i, j, k, :, :] = eigvecs_full[:, :6]

    print(f'Completed iteration {i+1} out of {len(Iy_vals)}')



np.save('param_spaces/param_space3D_2.npy', param_space)
np.save('param_spaces/param_space_eigvecs3D_2.npy', param_space_eigvecs)